# Part 5 - t-SNE Analizi ve Veri Görselleştirme

Projenin bu son aşamasında, 4. Bölümde `turkish-e5-large` modeli ile hesapladığımız 1024 boyutlu metin vektörlerini (embeddings) **t-SNE** algoritması kullanarak 2 boyutlu (2D) uzaya indirgiyoruz.

Amacımız, modelin doğru ve yanlış cevapladığı soruların anlamsal uzayda nasıl kümelendiğini görmek, ayrıca Claude ile ürettiğimiz 5 farklı soru stratejisinin orijinal sorulardan ne kadar uzaklaştığını görsel olarak analiz etmektir.

## 1. Kütüphanelerin Yüklenmesi ve Veri Setinin Hazırlanması
Gerekli görselleştirme ve makine öğrenmesi kütüphanelerini içe aktarıyoruz. Ardından orijinal ve üretilen (augmented) veri setlerimizi analiz için tek bir veri çerçevesinde (`df_all`) birleştiriyoruz.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.manifold import TSNE
from google.colab import drive

# Google Drive bağlantısı
drive.mount('/content/drive')

# Dosya Yolları
RESULTS_PATH          = "/content/drive/MyDrive/kaynaklar/augmented_dataset_with_results.csv"
ORIGINAL_PATH_SUCCESS = "/content/drive/MyDrive/kaynaklar/qwen3_5_4b_basarili_50.csv"
ORIGINAL_PATH_FAIL    = "/content/drive/MyDrive/kaynaklar/qwen3_5_4b_basarisiz_50.csv"
EMBEDDINGS_PATH       = "/content/drive/MyDrive/kaynaklar/embeddings.npy"

print("Veri setleri yükleniyor ve birleştiriliyor...")

# Orijinal 100 sorunun yüklenmesi
df_s = pd.read_csv(ORIGINAL_PATH_SUCCESS)
df_f = pd.read_csv(ORIGINAL_PATH_FAIL)
df_s['original_status'] = 'basarili'
df_f['original_status'] = 'basarisiz'

df_orig = pd.concat([df_s, df_f], ignore_index=True)
df_orig['qwen_correct'] = df_orig['original_status'].map({'basarili': True, 'basarisiz': False})
df_orig['soru_turu'] = 'orijinal'
df_orig['strategy']  = 'Orijinal'
df_orig['metin']     = df_orig['question']

# Üretilen 2500 sorunun yüklenmesi
df_aug = pd.read_csv(RESULTS_PATH)
df_aug['soru_turu'] = 'uretilen'
df_aug['metin']     = df_aug['generated_question']

# Verilerin tek çatıda birleştirilmesi
df_all = pd.concat(
    [df_orig[['metin', 'soru_turu', 'strategy', 'original_status', 'qwen_correct']],
     df_aug[['metin', 'soru_turu', 'strategy', 'original_status', 'qwen_correct']]],
    ignore_index=True
)

print(f"Veri seti hazır: Toplam {len(df_all)} satır.")

## 2. t-SNE Boyut İndirgeme
Önceki aşamada `.npy` formatında kaydettiğimiz yüksek boyutlu embedding matrisini yüklüyoruz. Scikit-learn kütüphanesindeki `TSNE` modülünü kullanarak bu verileri x ve y koordinatlarına (2 boyuta) dönüştürüyor ve ana veri çerçevemize ekliyoruz.

In [ ]:
print("Embedding vektörleri yükleniyor...")
embeddings = np.load(EMBEDDINGS_PATH)

print("t-SNE hesaplanıyor... (Bu işlem ~2-3 dakika sürebilir)")
tsne = TSNE(
    n_components=2,
    perplexity=40,
    n_iter=1000,
    random_state=42,
    verbose=1
)
tsne_result = tsne.fit_transform(embeddings)

# Hesaplanan 2D koordinatları dataframe'e ekliyoruz
df_all['tsne_x'] = tsne_result[:, 0]
df_all['tsne_y'] = tsne_result[:, 1]

print("t-SNE işlemi tamamlandı ve koordinatlar tabloya eklendi.")
print(df_all[['soru_turu', 'strategy', 'qwen_correct', 'tsne_x', 'tsne_y']].head())

## 3. Görselleştirme 1: Model Başarısı ve Strateji Dağılımı
İki alt grafik (subplot) oluşturuyoruz:
* **Sol Grafik:** Qwen modelinin doğru çözdüğü soruları yeşil, yanlış çözdüklerini kırmızı renkle gösterir. Orijinal sorular büyük yıldızlar ile vurgulanmıştır.
* **Sağ Grafik:** Soruların hangi strateji (Tema değişikliği, Karmaşıklaştırma vb.) ile üretildiğine göre renklendirildiği anlamsal kümelenme haritasıdır.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# ==========================================
# SOL GRAFİK: Doğru/Yanlış Dağılımı
# ==========================================
ax = axes[0]
renkler = {True: '#2ecc71', False: '#e74c3c', 'nan': '#cccccc'}

for dogru, grup in df_all.groupby(df_all['qwen_correct'].astype(str)):
    renk  = renkler.get(dogru, '#cccccc')
    alpha = 0.7 if dogru == 'True' else 0.5
    size  = 8 if grup['soru_turu'].iloc[0] == 'orijinal' else 4
    ax.scatter(grup['tsne_x'], grup['tsne_y'],
               c=renk, alpha=alpha, s=size, linewidths=0)

# Orijinal soruları büyük yıldız ile vurgula
for _, grup in df_all[df_all['soru_turu'] == 'orijinal'].groupby('qwen_correct'):
    renk = '#2ecc71' if grup['qwen_correct'].iloc[0] else '#e74c3c'
    ax.scatter(grup['tsne_x'], grup['tsne_y'],
               c=renk, s=120, marker='*', edgecolors='black', linewidths=0.5, zorder=5)

ax.set_title('t-SNE: Doğru / Yanlış Çözüm Dağılımı', fontsize=13)
legend_elements = [
    mpatches.Patch(color='#2ecc71', label='Doğru (üretilen)'),
    mpatches.Patch(color='#e74c3c', label='Yanlış (üretilen)'),
    plt.Line2D([0], [0], marker='*', color='w', markerfacecolor='#2ecc71',
               markeredgecolor='black', markersize=14, label='Doğru (orijinal)'),
    plt.Line2D([0], [0], marker='*', color='w', markerfacecolor='#e74c3c',
               markeredgecolor='black', markersize=14, label='Yanlış (orijinal)'),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')

# ==========================================
# SAĞ GRAFİK: Strateji Türüne Göre Dağılım
# ==========================================
ax = axes[1]
strateji_renkler = {
    'Orijinal':            '#000000',
    'Tema Değişikliği':    '#3498db',
    'Karmaşıklaştırma':    '#e67e22',
    'Sadeleştirme':        '#9b59b6',
    'Hikayeleştirme':      '#1abc9c',
    'Tersine Mühendislik': '#e74c3c',
}

for strateji, grup in df_all.groupby('strategy'):
    renk   = strateji_renkler.get(strateji, '#999999')
    size   = 80 if strateji == 'Orijinal' else 4
    marker = '*' if strateji == 'Orijinal' else 'o'
    zorder = 5 if strateji == 'Orijinal' else 1
    ax.scatter(grup['tsne_x'], grup['tsne_y'],
               c=renk, alpha=0.6, s=size, marker=marker,
               linewidths=0, zorder=zorder, label=strateji)

ax.set_title('t-SNE: Strateji Türüne Göre Dağılım', fontsize=13)
ax.legend(fontsize=10, markerscale=2)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')

plt.suptitle('Turkish-E5-Large Embeddings — GSM8K-TR Soru Uzayı Analizi', fontsize=16, y=1.02, fontweight='bold')
plt.tight_layout()

# Grafiği Kaydet ve Göster
plt.savefig('/content/drive/MyDrive/kaynaklar/tsne_analiz.png', dpi=150, bbox_inches='tight')
plt.show()
print("Analiz grafiği başarıyla kaydedildi.")

## 4. Görselleştirme 2: Kök Neden Analizi (Orijinal Soru Durumuna Göre)
Bu grafikte, sentetik (augmented) soruların başarısını, türetildikleri orijinal sorunun başarılı veya başarısız olmasına göre inceliyoruz. Acaba modelin aslen bilemediği sorulardan türetilen sorular da başarısızlık kümesinde mi kalıyor?